In [2]:
# notebook : sage_results_tables.ipynb — cellule 1 : tables de résultats SAGE à partir d'un hash d'évaluation (results.jsonl)
import hashlib
import json
from typing import Optional

import numpy as np
import pandas as pd
from omegaconf import OmegaConf
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

from src.config import RESULT_DIR
from src.data.dataset_aprdb import AprdbDataset

HASH = "580ae5ecf6642c478de600cb70fdba34b5ad7993ad7d52e2fa53f1e971f6bca7"
THRESHOLD = 0.5
ID_CANDIDATES = ["review-id", "custom_id", "id", "uid"]
META_ALIASES = {"conference": ["conference"], "year": ["year", "date"], "generator": ["generator", "model", "author"]}
SCALARS = ["auroc", "accuracy", "precision", "recall", "f1_score", "tpr_1_fpr", "tpr_5_fpr"]


def find_records(path, h: str) -> list[dict]:
    """Enregistrements de results.jsonl portant ce hash (filtre texte avant parsing : fichier potentiellement gros)."""
    out = []
    with open(path) as f:
        for line in f:
            if h in line:
                rec = json.loads(line)
                if rec.get("hash") == h:
                    out.append(rec)
    return out


def short(v, n: int = 200) -> str:
    s = repr(v)
    return s if len(s) <= n else s[:n] + "…"


def digest(text) -> str:
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()


def metrics(y: np.ndarray, p: np.ndarray) -> dict:
    """Mêmes définitions que le notebook récapitulatif : TPR interpolé sur la courbe ROC."""
    row = {"n": len(y), "n IA": int(y.sum()), "n hum.": int(len(y) - y.sum())}
    if len(np.unique(y)) < 2:
        return {**row, "AUC": np.nan, "TPR@1%": np.nan, "TPR@5%": np.nan, "acc": np.nan}
    fpr, tpr, _ = roc_curve(y, p)
    return {**row,
            "AUC": float(roc_auc_score(y, p)),
            "TPR@1%": float(np.interp(0.01, fpr, tpr)),
            "TPR@5%": float(np.interp(0.05, fpr, tpr)),
            "acc": float(((p >= THRESHOLD).astype(int) == y).mean())}


def by(cols: list[str]) -> pd.DataFrame:
    """Strates contenant les deux classes (humaines et IA de la même édition)."""
    rows = []
    for key, g in df.groupby(cols, sort=True):
        key = key if isinstance(key, tuple) else (key,)
        rows.append({**dict(zip(cols, key)), **metrics(g["y_true"].to_numpy(), g["y_score"].to_numpy())})
    return pd.DataFrame(rows).set_index(cols)


def by_generator(within: Optional[list[str]] = None) -> pd.DataFrame:
    """IA d'un générateur contre TOUTES les humaines de la strate : ne dépend pas du champ générateur des humaines."""
    within = within or []
    groups = df.groupby(within, sort=True) if within else [((), df)]
    rows = []
    for key, g in groups:
        key = key if isinstance(key, tuple) else (key,)
        hum = g[g["y_true"] == 0]
        for gen, ai in g[g["y_true"] == 1].groupby("generator", sort=True):
            part = pd.concat([ai, hum])
            rows.append({**dict(zip(within, key)), "generator": gen,
                         **metrics(part["y_true"].to_numpy(), part["y_score"].to_numpy())})
    return pd.DataFrame(rows).set_index(within + ["generator"])


def show(table: pd.DataFrame, title: str) -> None:
    print(f"\n=== {title} ===")
    print(table.to_string(float_format=lambda x: f"{x:.4f}"))


# ------------------------------------------------------------------ 0. enregistrement du run
records = find_records(RESULT_DIR / "results.jsonl", HASH)
if not records:
    raise LookupError(f"aucun enregistrement pour {HASH} dans {RESULT_DIR / 'results.jsonl'}")
if len(records) > 1:
    print(f"⚠️ {len(records)} enregistrements pour ce hash : utilisation du dernier")
rec = records[-1]

print(f"=== 0. Run {HASH[:12]}… ===")
print("métriques enregistrées :", {k: rec.get(k) for k in SCALARS})
for prefix in ("dataset.", "dataset_train.", "dataset_val.", "training.", "detector."):
    sub = {k[len(prefix):]: short(v) for k, v in rec.items() if k.startswith(prefix)}
    print(f"--- {prefix[:-1]} ---")
    for k, v in sub.items():
        print(f"  {k}: {v}")

# ------------------------------------------------------------------ prédictions
scores = np.asarray(rec["y_scores"], dtype=float).reshape(len(rec["y_scores"]), -1)
if scores.shape[1] == 1:
    scores = scores[:, 0]
elif scores.shape[1] == 2:
    scores = scores[:, 1]
else:
    raise ValueError(f"y_scores à {scores.shape[1]} colonnes : ce script couvre le cas binaire")
pred = pd.DataFrame({"id": [str(i) for i in rec["ids"]], "content": rec["content"],
                     "y_true": np.asarray(rec["y_true"], dtype=int).ravel(), "y_score": scores})
print(f"\nprédictions : {len(pred)} | ids uniques {pred['id'].nunique()} | exemples {pred['id'].head(3).tolist()}")

# ------------------------------------------------------------------ métadonnées : rechargement du dataset de test
ds_cfg = OmegaConf.create({k[len("dataset."):]: v for k, v in rec.items() if k.startswith("dataset.")})
ds = AprdbDataset(config=ds_cfg)
meta = ds.to_pandas() if hasattr(ds, "to_pandas") else pd.DataFrame([ds[i] for i in range(len(ds))])
for canon, cands in META_ALIASES.items():
    if canon not in meta.columns:
        found = next((c for c in cands if c in meta.columns), None)
        if found is None:
            raise KeyError(f"colonne {canon!r} introuvable | colonnes du dataset : {list(meta.columns)}")
        meta[canon] = meta[found].astype(str).str[:4] if canon == "year" and found == "date" else meta[found]

pred_ids = set(pred["id"])
cover = {c: len(pred_ids & set(meta[c].astype(str))) for c in ID_CANDIDATES if c in meta.columns}
if not cover or max(cover.values()) == 0:
    raise KeyError(f"aucune colonne d'identifiant du dataset ne recoupe les ids | couverture : {cover}")
id_col = max(cover, key=cover.get)
print(f"colonne d'identifiant : {id_col!r} (couverture {cover})")

meta["_c"] = meta["content"].map(digest)
meta["_k"] = meta[id_col].astype(str) + "|" + meta["_c"]
pred["_c"] = pred["content"].map(digest)
pred["_k"] = pred["id"] + "|" + pred["_c"]
cols = ["conference", "year", "generator"]

df = pred.merge(meta.drop_duplicates("_k")[["_k"] + cols], on="_k", how="left", validate="many_to_one")
unmatched = df["conference"].isna()
if unmatched.any():   # repli : texte seul, si ce texte est unique dans le dataset
    uniq = meta[~meta["_c"].duplicated(keep=False)].set_index("_c")[cols]
    fill = df.loc[unmatched, "_c"].map(lambda c: c in uniq.index)
    for c in cols:
        df.loc[unmatched & fill.reindex(df.index, fill_value=False), c] = \
            df.loc[unmatched & fill.reindex(df.index, fill_value=False), "_c"].map(uniq[c])
still = df["conference"].isna().sum()
print(f"appariement : {len(df) - unmatched.sum()} par (id, texte) | {unmatched.sum() - still} par texte seul | "
      f"{still} non appariées")
if still:
    raise ValueError(f"{still} prédictions sans métadonnées : exemples {df.loc[df['conference'].isna(), 'id'].head(5).tolist()}")

# ------------------------------------------------------------------ 1. global + contrôle contre l'enregistrement
glob = metrics(df["y_true"].to_numpy(), df["y_score"].to_numpy())
show(pd.DataFrame([glob], index=["global"]), "1. Global")
for mine, theirs in (("AUC", "auroc"), ("acc", "accuracy"), ("TPR@1%", "tpr_1_fpr"), ("TPR@5%", "tpr_5_fpr")):
    print(f"recalculé {mine} = {glob[mine]:.4f} | enregistré {theirs} = {short(rec.get(theirs), 40)}")
print("\n" + classification_report(df["y_true"], (df["y_score"] >= THRESHOLD).astype(int), digits=3,
                                   target_names=["human (0)", "AI (1)"]))

# ------------------------------------------------------------------ 2. strates marginales
show(by(["conference"]), "2a. Par conférence")
show(by(["year"]), "2b. Par année (conférences regroupées)")
show(by(["conference", "year"]), "2c. Par conférence et année")
show(by_generator(), "2d. Par générateur (IA du générateur vs toutes les humaines)")

# ------------------------------------------------------------------ 3. grilles générateur × édition
grid = by_generator(["conference", "year"])
show(grid["AUC"].unstack("generator"), "3a. AUC, générateur × édition")
show(grid["n IA"].unstack("generator").astype("Int64"), "3b. Effectifs IA, générateur × édition")
show(by_generator(["year"])["AUC"].unstack("generator"), "3c. AUC, générateur × année (conférences regroupées)")

/data2/cyaiche/cache_dir
⚠️ 4 enregistrements pour ce hash : utilisation du dernier
=== 0. Run 580ae5ecf664… ===
métriques enregistrées : {'auroc': 0.964228616636528, 'accuracy': 0.9000424448217318, 'precision': 0.8961343225302616, 'recall': 0.918, 'f1_score': 0.9069353882631891, 'tpr_1_fpr': 0.5464, 'tpr_5_fpr': 0.8144}
--- dataset ---
  name: 'aprdb'
  description: 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt'
  conference: None
  date: None
  keep_meta: True
  subset_size: 10000
  split: 'test'
  with_pdf: False
  seed: 42
--- dataset_train ---
  name: 'aprdb'
  description: 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt'
  conference: None
  date: None
  keep_meta: True
  subset_size: 10000
  split: 'calibration'
  with_pdf: False
  seed: 42
--- dataset_val ---
  name: 'aprdb'
  description: 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt'
  conference: None
  date: None
  keep_meta: True
  subset_size: 10000
  split: 'test'

In [4]:
# notebook : sage_results_tables.ipynb — cellule 2 : accuracy maximale sur la courbe ROC (à exécuter après la cellule 1)
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

COLS = ["AUC", "TPR@1%", "TPR@5%", "acc", "acc max", "seuil acc", "bacc max", "seuil bacc"]


def max_acc(y: np.ndarray, p: np.ndarray) -> dict:
    """Accuracy et accuracy équilibrée maximales sur tous les seuils de la courbe ROC (seuil oracle).
    drop_intermediate=False : tous les seuils distincts sont examinés."""
    if len(np.unique(y)) < 2:
        return {"acc max": np.nan, "seuil acc": np.nan, "bacc max": np.nan, "seuil bacc": np.nan}
    fpr, tpr, thr = roc_curve(y, p, drop_intermediate=False)
    pos = int(y.sum())
    neg = len(y) - pos
    acc = (tpr * pos + (1 - fpr) * neg) / (pos + neg)
    bacc = (tpr + 1 - fpr) / 2
    i, j = int(acc.argmax()), int(bacc.argmax())
    return {"acc max": float(acc[i]), "seuil acc": float(thr[i]),
            "bacc max": float(bacc[j]), "seuil bacc": float(thr[j])}


def row(g: pd.DataFrame) -> dict:
    y_, p_ = g["y_true"].to_numpy(), g["y_score"].to_numpy()
    return {**metrics(y_, p_), **max_acc(y_, p_)}


def table_generator() -> pd.DataFrame:
    """IA d'un générateur contre toutes les humaines (même définition que la table 2)."""
    hum = df[df["y_true"] == 0]
    rows = [{"generator": gen, **row(pd.concat([ai, hum]))}
            for gen, ai in df[df["y_true"] == 1].groupby("generator", sort=True)]
    rows.append({"generator": "All", **row(df)})
    return pd.DataFrame(rows).set_index("generator")


def table_edition() -> pd.DataFrame:
    """Humaines et IA d'une même édition, plus une ligne All par conférence (même définition que la table 3)."""
    rows = [{"conference": c, "year": str(y), **row(g)}
            for (c, y), g in df.groupby(["conference", "year"], sort=True)]
    rows += [{"conference": c, "year": "All", **row(g)} for c, g in df.groupby("conference", sort=True)]
    out = pd.DataFrame(rows)
    out["_last"] = out["year"].eq("All")
    return out.sort_values(["conference", "_last", "year"]).drop(columns="_last").set_index(["conference", "year"])


t_gen, t_ed = table_generator(), table_edition()
show(t_gen[COLS], "Table 2 : par générateur (IA du générateur vs toutes les humaines)")
show(t_ed[COLS], "Table 3 : par conférence et année")

# contrôle : l'optimum ne peut pas être inférieur à la valeur au seuil 0,5
for name, t in (("table 2", t_gen), ("table 3", t_ed)):
    bad = t.index[t["acc max"] < t["acc"] - 1e-12].tolist()
    print(f"{'✅' if not bad else '⚠️'} {name} : acc max ≥ acc au seuil {THRESHOLD} {bad if bad else ''}")


=== Table 2 : par générateur (IA du générateur vs toutes les humaines) ===
             AUC  TPR@1%  TPR@5%    acc  acc max  seuil acc  bacc max  seuil bacc
generator                                                                        
claude    0.9755  0.6393  0.8849 0.8918   0.9429     0.8533    0.9210      0.7187
gemini    0.9604  0.5515  0.8206 0.8843   0.9278     0.8666    0.8970      0.6006
gpt4o     0.9766  0.5770  0.8733 0.8940   0.9407     0.8553    0.9264      0.6372
llama     0.9309  0.2842  0.6030 0.8693   0.8865     0.7243    0.8589      0.3587
qwen      0.9788  0.6844  0.8955 0.8946   0.9450     0.8822    0.9270      0.6471
All       0.9642  0.5464  0.8144 0.9000   0.9012     0.5534    0.9008      0.5832

=== Table 3 : par conférence et année ===
                   AUC  TPR@1%  TPR@5%    acc  acc max  seuil acc  bacc max  seuil bacc
conference year                                                                        
ICLR       2017 0.9563  0.3426  0.8194 0.8829   0